In [ ]:
import os
import subprocess
import time

os.chdir('/workspace')

print('正在安装基础依赖...')
subprocess.run(['apt-get', 'update', '-y'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'ffmpeg'], check=True)

print('正在下载 AI 视频工具...')
if not os.path.exists('MoneyPrinterTurbo'):
    subprocess.run(['git', 'clone', 'https://github.com/harry0703/MoneyPrinterTurbo.git'], check=True)
if not os.path.exists('NarratoAI'):
    subprocess.run(['git', 'clone', 'https://github.com/linyqh/NarratoAI.git'], check=True)

print('正在安装 Python 依赖...')
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-i', 'https://mirrors.aliyun.com/pypi/simple/'], cwd='MoneyPrinterTurbo', check=True)
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-i', 'https://mirrors.aliyun.com/pypi/simple/'], cwd='NarratoAI', check=True)

print('正在启动服务...')
# 启动 NarratoAI (8501)
subprocess.Popen(['nohup', 'streamlit', 'run', 'webui.py', '--server.port', '8501', '--server.maxUploadSize', '2048'], cwd='/workspace/NarratoAI', stdout=open('/workspace/NarratoAI/narrato.log', 'w'), stderr=subprocess.STDOUT)
# 启动 MoneyPrinterTurbo (8502)
subprocess.Popen(['nohup', 'streamlit', 'run', './webui/Main.py', '--server.port', '8502'], cwd='/workspace/MoneyPrinterTurbo', stdout=open('/workspace/MoneyPrinterTurbo/mpt.log', 'w'), stderr=subprocess.STDOUT)
# 启动 cloudflared 隧道
subprocess.Popen(['nohup', './cloudflared', 'tunnel', '--url', 'http://localhost:8501'], cwd='/workspace', stdout=open('/workspace/tunnel_8501.log', 'w'), stderr=subprocess.STDOUT)
subprocess.Popen(['nohup', './cloudflared', 'tunnel', '--url', 'http://localhost:8502'], cwd='/workspace', stdout=open('/workspace/tunnel_8502.log', 'w'), stderr=subprocess.STDOUT)

print('正在等待隧道启动...')
time.sleep(10)

print('\nNarratoAI 访问地址:')
subprocess.run(['grep', '-o', 'https://[a-z0-9-]*\.trycloudflare\.com', '/workspace/tunnel_8501.log'], check=True)
print('\nMoneyPrinterTurbo 访问地址:')
subprocess.run(['grep', '-o', 'https://[a-z0-9-]*\.trycloudflare\.com', '/workspace/tunnel_8502.log'], check=True)

print('\n✅ 所有服务已启动！笔记本将持续运行以维持服务。')
# 保持笔记本运行
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print('已停止')
